<a href="https://colab.research.google.com/github/nhanle1992/Rock-paper-scissors-Game/blob/master/practice_salary_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade numpy pandas matplotlib seaborn scikit-learn scipy statsmodels plotly requests beautifulsoup4 openpyxl lxml


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 43.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing ins

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive is mounted and library is installed!")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Read the CSV file into a DataFrame, ensuring a fresh, unscaled load
salary_df = pd.read_csv('/content/drive/MyDrive/Dataset/Salary/Salary_dataset.csv')

# Explicitly drop 'Unnamed: 0' and 'YearsExperience_squared' if they exist to ensure a clean slate
if 'Unnamed: 0' in salary_df.columns:
    salary_df = salary_df.drop('Unnamed: 0', axis=1)
if 'YearsExperience_squared' in salary_df.columns:
    salary_df = salary_df.drop('YearsExperience_squared', axis=1)

print("CSV file reloaded and cleaned successfully to ensure unscaled and un-engineered data!")
display(salary_df.head())

In [ ]:
salary_df.info()

In [ ]:
salary_df.dtypes

In [ ]:
salary_df.head()

In [ ]:
salary_df.tail()

In [ ]:
# 1. Total missing values per column (Best for a quick sanity check)
salary_df.isna().sum()

In [ ]:
#Grand total of all missing values across the entire dataset
salary_df.isna().sum().sum()


In [ ]:
#Percentage of missing values in each column
(salary_df.isna().sum() / len(salary_df)) * 100

In [ ]:
# A hypothetical dataset dictionary for illustration, as one was not provided with the raw data.
dataset_dictionary = {
    'YearsExperience': {
        'Description': 'Number of years of professional experience',
        'Type': 'float64',
        'Range': '1.2 to 10.6 years'
    },
    'Salary': {
        'Description': 'Annual salary of the individual',
        'Type': 'float64',
        'Range': '$37,732 to $122,392'
    },
    'YearsExperience_squared': {
        'Description': 'Squared value of YearsExperience, created as a feature engineering step',
        'Type': 'float64',
        'Range': 'Derived from YearsExperience'
    }
}

print("Dataset Dictionary:")
display(pd.DataFrame.from_dict(dataset_dictionary, orient='index'))

In [ ]:
# Count total number of duplicate rows
salary_df.duplicated().sum()

In [ ]:
# 2. View the actual duplicate rows (to inspect them manually)
salary_df[salary_df.duplicated()]

In [ ]:
# 3. Check duplicates based on specific columns (e.g., duplicate User IDs)
salary_df.duplicated(subset=['YearsExperience']).sum()
salary_df.duplicated(subset=['Salary']).sum()


In [ ]:
salary_df.describe()

In [ ]:
#Text Summary: Shows unique counts and top values for 'object' (text) columns
try:
    obj_description = salary_df.describe(include='object')
    if not obj_description.empty:
        display(obj_description)
    else:
        print("No object (string) columns found in the DataFrame to describe.")
except ValueError as e:
    print(f"An error occurred while describing object columns: {e}")
    print("This usually happens if there are no 'object' type columns in the DataFrame.")


In [ ]:
# Complete Summary: Combines both numerical and text columns into one view
salary_df.describe(include='all').T

## Data Cleaning: Handling Redundant Columns

Often, datasets come with columns that are just row identifiers (like an index) and don't provide analytical value. We'll check for such columns, like 'Unnamed: 0', and remove them if they are indeed redundant.

This code block checks for and removes a potentially redundant 'Unnamed: 0' column from the `salary_df` DataFrame. This column often appears as an artifact when reading CSV files that already contain an index column, and it can be safely removed if it duplicates the DataFrame's default index.

In [ ]:
unique_counts = salary_df.nunique()
print(unique_counts)

In [ ]:
constant_cols = unique_counts[unique_counts == 1].index.tolist()
print(constant_cols)

In [ ]:
# Check if 'Unnamed: 0' is just an index and can be dropped
if 'Unnamed: 0' in salary_df.columns:
    if (salary_df['Unnamed: 0'] == salary_df.index).all():
        print("'Unnamed: 0' column is a redundant index and will be dropped.")
        salary_df = salary_df.drop('Unnamed: 0', axis=1)
    else:
        print("'Unnamed: 0' column contains unique values and might be meaningful, not dropping.")
else:
    print("'Unnamed: 0' column not found.")

display(salary_df.head())

## Univariate Analysis

Univariate analysis involves examining each variable in the dataset individually to understand its distribution, central tendency, and spread. We'll look at key statistics and visualize the distributions for 'YearsExperience' and 'Salary'.

This cell calculates and displays the descriptive statistics (count, mean, standard deviation, min, max, and quartiles) for the numerical columns 'YearsExperience' and 'Salary' in the `salary_df` DataFrame. This helps to understand the central tendency, dispersion, and shape of each variable's distribution.

In [ ]:
# Descriptive Statistics for numerical columns
print("\nDescriptive statistics for YearsExperience:")
display(salary_df['YearsExperience'].describe())

print("\nDescriptive statistics for Salary:")
display(salary_df['Salary'].describe())

### Visualizing YearsExperience Distribution

This code block generates a histogram with a Kernel Density Estimate (KDE) and a box plot for the 'YearsExperience' column. These visualizations help to assess the distribution, identify skewness, and detect potential outliers in the years of experience data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of YearsExperience
plt.figure(figsize=(10, 6))
sns.histplot(salary_df['YearsExperience'], kde=True, bins=5)
plt.title('Distribution of YearsExperience')
plt.xlabel('YearsExperience')
plt.ylabel('Frequency')
plt.show()

# Box plot of YearsExperience
plt.figure(figsize=(10, 2))
sns.boxplot(x=salary_df['YearsExperience'])
plt.title('Box Plot of YearsExperience')
plt.xlabel('YearsExperience')
plt.show()

#### Interpretation of YearsExperience Charts for `salary_df`

*   **Histogram:** The histogram for 'YearsExperience' in your dataset shows the frequency distribution of experience levels, ranging from approximately **1.2 to 10.6 years**. The `kde` (Kernel Density Estimate) curve suggests a relatively **even distribution across this range**, with no extreme peaks or valleys, though there might be a slight tendency for more individuals in the lower-mid experience bracket. This indicates a good spread of experience among the individuals in your `salary_df`, meaning you have a diverse group of employees from entry-level to experienced. This even distribution is generally good for building predictive models, as it ensures there's enough data across the entire spectrum of 'YearsExperience' to learn from.
*   **Box Plot:** The box plot visually summarizes the 'YearsExperience' data. The central box represents the interquartile range (IQR), with the line inside indicating the median. The 'whiskers' extend to show the range of the data. In this plot, there are **no apparent outliers**, and the distribution appears to be fairly symmetrical or only slightly skewed, which aligns with the histogram's observations. The lack of outliers suggests that all recorded 'YearsExperience' values are within expected bounds for the dataset, simplifying further analysis as no special handling for extreme experience values is immediately required.

### Visualizing Salary Distribution

Similar to the previous cell, this code block creates a histogram with KDE and a box plot for the 'Salary' column. These plots provide a visual representation of the salary distribution, allowing us to observe its shape, spread, and any unusual data points.

In [ ]:
# Histogram of Salary
plt.figure(figsize=(10, 6))
sns.histplot(salary_df['Salary'], kde=True, bins=10)
plt.title('Distribution of Salary')
plt.xlabel('Salary')
plt.ylabel('Frequency')
plt.show()

# Box plot of Salary
plt.figure(figsize=(10, 2))
sns.boxplot(x=salary_df['Salary'])
plt.title('Box Plot of Salary')
plt.xlabel('Salary')
plt.show()

#### Interpretation of Salary Charts for `salary_df`

*   **Histogram:** The histogram for 'Salary' in your `salary_df` reveals its frequency distribution, with salaries ranging from approximately **$37,732 to $122,392**. The `kde` curve **clearly indicates a right-skewed distribution**. This means there are more employees earning lower salaries, and fewer employees earning very high salaries within your dataset. This pattern is typical for salary data where a few higher earners can pull the average up. This skewness is important to note for modeling, as linear models might assume normally distributed residuals, and this distribution might suggest transformation of the 'Salary' variable (e.g., log transformation) could be beneficial for better model performance or interpretability.
*   **Box Plot:** The box plot for 'Salary' further **confirms the right-skewness** observed in the histogram. The median line within the box is positioned closer to the lower quartile, and the upper whisker is longer than the lower one. This visually reinforces the idea that there's a longer tail of higher salaries. Despite this skewness, the plot shows **no extreme outliers** in the salary data, suggesting all values are within a reasonable range for this dataset. The wide range and skewness imply that predicting 'Salary' might be more challenging for higher values, and the model might perform better on the majority of lower to mid-range salaries.

## Bivariate Analysis: YearsExperience vs. Salary

Bivariate analysis examines the relationship between two variables. Here, we'll explore how 'YearsExperience' relates to 'Salary' using a scatter plot and correlation coefficient.

### Scatter Plot: YearsExperience vs. Salary

A scatter plot is ideal for visualizing the relationship between two numerical variables. It helps us observe the pattern, direction (positive or negative), and strength of their association.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.scatterplot(x='YearsExperience', y='Salary', data=salary_df)
plt.title('Scatter Plot of YearsExperience vs. Salary')
plt.xlabel('Years of Experience')
plt.ylabel('Salary')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

#### Interpretation of Scatter Plot

*   **Direction:** The scatter plot for 'YearsExperience' and 'Salary' clearly shows an **upward trend**. As 'YearsExperience' increases, 'Salary' also tends to increase. This indicates a positive relationship between the two variables.
*   **Form:** The relationship appears to be **linear**. The data points roughly follow a straight line, suggesting that for each additional year of experience, there is a somewhat consistent increase in salary.
*   **Strength:** The data points are relatively **tightly clustered around an imaginary line**, indicating a strong positive relationship. There isn't a lot of scatter, which means 'YearsExperience' is a good predictor of 'Salary' in this dataset.
*   **Outliers:** Visually, there are **no apparent extreme outliers** that deviate significantly from the overall linear trend, which is a good sign for modeling purposes.

### Correlation Coefficient: YearsExperience and Salary

The Pearson correlation coefficient quantifies the linear relationship between two variables. Its value ranges from -1 to 1, where:
*   1 indicates a perfect positive linear relationship.
*   -1 indicates a perfect negative linear relationship.
*   0 indicates no linear relationship.

In [ ]:
correlation = salary_df['YearsExperience'].corr(salary_df['Salary'])
print(f"Pearson Correlation Coefficient between YearsExperience and Salary: {correlation:.4f}")

#### Interpretation of Correlation Coefficient

The calculated Pearson Correlation Coefficient is approximately **0.9782**. A value close to **1** confirms a **very strong positive linear relationship** between 'YearsExperience' and 'Salary'. This means that as an individual's years of experience increase, their salary tends to increase proportionally and predictably. This high positive correlation is highly desirable for predictive modeling, as 'YearsExperience' is a significant factor in determining 'Salary' within this dataset.

## Multivariate Analysis

Multivariate analysis involves analyzing multiple variables simultaneously to understand their relationships and interactions. Although our dataset is relatively simple with only two main numerical variables ('YearsExperience' and 'Salary'), we can still explore their combined behavior using tools like a correlation matrix heatmap and a pair plot.

### Correlation Matrix Heatmap

A correlation matrix displays the Pearson correlation coefficients between all pairs of numerical variables in a dataset. A heatmap is a visual representation of this matrix, making it easy to spot strong and weak correlations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate the correlation matrix
correlation_matrix = salary_df[['YearsExperience', 'Salary']].corr()

# Plotting the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix Heatmap of YearsExperience and Salary')
plt.show()

#### Interpretation of Correlation Matrix Heatmap

The heatmap visually confirms the strong positive linear relationship between 'YearsExperience' and 'Salary'. The correlation coefficient of **0.98** (rounded to two decimal places) is highlighted, indicating that as one variable increases, the other tends to increase almost perfectly linearly. The `coolwarm` colormap further emphasizes this: a warm color (like red) suggests a strong positive correlation. This visualization reinforces our earlier finding from the bivariate analysis, solidifying the idea that these two variables move in tandem.

### Pair Plot

A pair plot (or scatterplot matrix) is a grid of plots where each variable in the dataset is plotted against every other variable. It shows the distribution of each variable (typically a histogram or KDE plot) on the diagonal, and the scatter plots of the relationships between pairs of variables in the off-diagonal. This provides a quick overview of all pairwise relationships and distributions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a pair plot
plt.figure(figsize=(10, 8))
sns.pairplot(salary_df[['YearsExperience', 'Salary']], diag_kind='kde')
plt.suptitle('Pair Plot of YearsExperience and Salary', y=1.02) # Adjust suptitle position
plt.show()

#### Interpretation of Pair Plot

The pair plot provides a concise view of both the univariate distributions and the bivariate relationship.

*   **Diagonal Plots (Distributions):** The plots on the diagonal show the Kernel Density Estimates (KDEs) for 'YearsExperience' and 'Salary' individually.
    *   For 'YearsExperience', the KDE shows a relatively uniform distribution, as we observed in the univariate analysis, indicating a consistent spread of experience levels.
    *   For 'Salary', the KDE confirms the right-skewed distribution, with a higher density of lower salaries and a tapering off towards higher salaries.

*   **Off-Diagonal Plots (Bivariate Relationships):** The off-diagonal plots show the scatter plot of 'YearsExperience' versus 'Salary' (and vice versa, though they are mirror images). These plots clearly demonstrate the **strong positive linear relationship** we've identified. The points cluster closely around an upward-sloping line, reinforcing the high correlation coefficient. This visualization further confirms that as years of experience increase, salary consistently rises, making 'YearsExperience' a very strong predictor of 'Salary' in this dataset.

## Data Pre-processing (Revisited)

We've already covered some key pre-processing steps during our initial data exploration:

*   **Missing Values:** We confirmed there are **no missing values** in our dataset, so no imputation strategies are needed.
*   **Duplicate Values:** We also verified that there are **no duplicate rows**, ensuring the uniqueness of our observations.
*   **One-Hot Encoding:** As discussed, our dataset (`salary_df`) currently only contains numerical features ('YearsExperience' and 'Salary'). One-hot encoding is specifically used for converting **categorical (non-numerical) features** into a numerical format suitable for machine learning models. Since we don't have any categorical features, **one-hot encoding is not applicable or necessary** for this dataset.

## Model Building: Train-Test Split

Before we build any machine learning model, it's crucial to split our dataset into two main parts: a **training set** and a **testing set**.

### Why do we split the data?

Imagine you're studying for an exam. If you only study the exact questions that will be on the test, you might get a perfect score, but that doesn't mean you truly understand the subject. You've just memorized the answers.

In machine learning:
*   The **training set** is like your study material. We use this data to teach the model to find patterns and relationships between features (inputs) and the target (output).
*   The **testing set** is like the actual exam. This data is kept completely separate and is **unseen** by the model during training. After the model is trained, we use the testing set to evaluate how well it performs on new data it has never encountered before. This gives us an honest assessment of the model's ability to generalize, rather than just memorizing the training data.

### 1. Define Features (X) and Target (y)

First, we need to clearly identify which columns in our `salary_df` will be used as **features (inputs)** and which column is our **target (output)** that we want to predict.

*   **Features (X):** These are the independent variables that our model will use to make predictions. In our case, this is 'YearsExperience'.
*   **Target (y):** This is the dependent variable we are trying to predict, which is 'Salary'.

In [ ]:
import pandas as pd

# Define features (X) - independent variables. Starting with only YearsExperience for the baseline model.
# Ensure X is derived from the unscaled salary_df.
X = salary_df[['YearsExperience']]

# Define target (y) - dependent variable
y = salary_df['Salary']

print("Features (X) head (should now be unscaled):")
display(X.head())

print("\nTarget (y) head:")
display(y.head())

#### Interpretation of Features (X) and Target (y)

We've separated our dataset into `X` (our input features) and `y` (our output target). This is a standard practice that makes it easy to feed the correct data to machine learning models. `X` now holds the scaled 'YearsExperience' values, which the model will learn from, and `y` holds the corresponding 'Salary' values that the model will try to predict.

### 2. Perform the Train-Test Split

Now, we'll use the `train_test_split` function from scikit-learn to divide `X` and `y` into their respective training and testing sets.

*   `X_train`, `y_train`: Used for training the model.
*   `X_test`, `y_test`: Used for evaluating the trained model.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
# test_size=0.2 means 20% of the data will be used for testing, and 80% for training.
# random_state=42 ensures reproducibility. If you run this code multiple times, you'll get the same split.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shapes of the split datasets:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

print("\nColumns in X_train:")
print(X_train.columns)
print("Columns in X_test:")
print(X_test.columns)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data ONLY and transform X_train
X_train_scaled = scaler.fit_transform(X_train)

# Transform X_test using the scaler fitted on X_train
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully after train-test split!")

print("\nFirst 5 rows of X_train_scaled:")
display(pd.DataFrame(X_train_scaled, columns=X_train.columns).head())

print("\nFirst 5 rows of X_test_scaled:")
display(pd.DataFrame(X_test_scaled, columns=X_test.columns).head())

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data ONLY and transform X_train
X_train_scaled = scaler.fit_transform(X_train)

# Transform X_test using the scaler fitted on X_train
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully after train-test split!")

print("\nFirst 5 rows of X_train_scaled:")
display(pd.DataFrame(X_train_scaled, columns=X_train.columns).head())

print("\nFirst 5 rows of X_test_scaled:")
display(pd.DataFrame(X_test_scaled, columns=X_test.columns).head())

#### Interpretation of Train-Test Split

We've now successfully divided our data!

*   `X_train` and `y_train` contain **80%** of the original data (24 samples out of 30) and will be used to teach our model. The model will look at these 24 experience/salary pairs and learn the relationship between them.
*   `X_test` and `y_test` contain the remaining **20%** of the data (6 samples out of 30) and will be used to evaluate the model's performance. Since the model has never seen this data during training, its performance on `X_test` will give us a good indication of how well it can predict salaries for new, real-world individuals.

Setting `random_state=42` is like shuffling a deck of cards in a specific way before splitting; it ensures that every time we run this code, the split is exactly the same, which is vital for comparing different models or analyses reliably. The `test_size=0.2` means that 20% of the data was reserved for testing, and 80% for training. This is a common split ratio.

## 3. Model Training: Linear Regression

**Linear Regression** is a fundamental supervised learning algorithm used for predicting a continuous target variable (like Salary) based on one or more input features (like Years of Experience).

The goal of a linear regression model is to find the best-fitting straight line (or hyperplane in higher dimensions) that describes the relationship between the input features and the target variable. This line is represented by the equation:

$y = b_0 + b_1x_1 + b_2x_2 + ... + b_nx_n$

Where:
*   $y$ is the predicted target variable (Salary).
*   $b_0$ is the intercept (the predicted salary when all experience is zero).
*   $b_1, b_2, ..., b_n$ are the coefficients (slopes) for each feature $x_1, x_2, ..., x_n$.
*   $x_1, x_2, ..., x_n$ are the input features (YearsExperience, YearsExperience_squared).

During training, the model learns the optimal values for $b_0$ and the coefficients ($b_1, ..., b_n$) that minimize the difference between its predictions and the actual `y_train` values.

In [ ]:
from sklearn.linear_model import LinearRegression

# Initialize the Linear Regression model
model = LinearRegression()

# Train the model using the scaled training data
# The .fit() method is where the model learns from X_train_scaled and y_train
model.fit(X_train_scaled, y_train)

print("Linear Regression model trained successfully!")

# Display the learned coefficients and intercept
print(f"\nModel Intercept: {model.intercept_:.2f}")
print(f"Model Coefficients (for YearsExperience): {model.coef_.round(2)}")

#### Interpretation of Model Training

We've now trained our Linear Regression model! The `.fit()` method is where the machine learning magic happens. The model looked at all the `YearsExperience` values in `X_train` and their corresponding `Salary` values in `y_train`, and it figured out the best line to represent this relationship.

*   **Intercept ($b_0$):** This is the base salary the model predicts for someone with zero years of experience (after scaling). In our case, the value **`74208.62`** means that, if the scaled `YearsExperience` were 0, the predicted salary would be around $74,208.62. This is the starting point of our prediction.

*   **Coefficient ($b_1$):** This value tells us how much the salary is expected to change for a one-unit increase in the 'YearsExperience' feature. For example, the coefficient (**`27151.54`**) relates to the `YearsExperience` (scaled). The positive coefficient for `YearsExperience` suggests that salary generally increases with experience.

## 4. Model Validation: Predictions and Evaluation

After training our model, we need to check how well it performs on data it has *not* seen before. This is where our `X_test` and `y_test` datasets come into play.

### 4.1 Making Predictions

We'll use our trained `model` to predict the `Salary` values for the `X_test` features. These are our model's best guesses for what the salaries should be.

In [ ]:
# Make predictions on the scaled test set
y_pred = model.predict(X_test_scaled)

print("First 5 actual salaries from test set (y_test):")
display(y_test.head())

print("\nFirst 5 predicted salaries from test set (y_pred):")
display(pd.Series(y_pred).head())

#### Interpretation of Predictions

We now have two sets of salaries for our test data:
*   **`y_test`**: These are the actual salaries that we know to be true for the individuals in our test set.
*   **`y_pred`**: These are the salaries that our trained Linear Regression model *predicted* for those same individuals, based on their `YearsExperience` and `YearsExperience_squared` values.

Our goal in the next step is to compare `y_pred` with `y_test` to see how close our predictions are to the actual values.

### 4.2 Model Evaluation

To quantitatively assess our model's performance, we use **evaluation metrics**. These metrics provide numerical summaries of how well our predictions (`y_pred`) match the actual values (`y_test`). For regression tasks, common metrics include:

*   **Mean Squared Error (MSE):** This measures the average of the squares of the errors (the difference between predicted and actual values). It gives more weight to larger errors.
    *   *Formula:* $MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$
*   **Root Mean Squared Error (RMSE):** This is the square root of MSE. It's often preferred because it's in the same units as the target variable, making it easier to interpret.
    *   *Formula:* $RMSE = \sqrt{MSE}$
*   **R-squared ($R^2$):** This represents the proportion of the variance in the dependent variable that is predictable from the independent variables. It ranges from 0 to 1, where 1 indicates a perfect fit.
    *   *Formula:* $R^2 = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Calculate Mean Squared Error (MSE)
mse = mean_squared_error(y_test, y_pred)

# Calculate Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

# Calculate R-squared (R2 Score)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R2 Score): {r2:.4f}")

#### Interpretation of Model Evaluation Metrics

Let's break down what these numbers tell us about our model's performance:

*   **Mean Squared Error (MSE) and Root Mean Squared Error (RMSE):** These values indicate the average magnitude of the errors. A lower MSE/RMSE means the model's predictions are, on average, closer to the actual values. Since RMSE is in the same units as our target variable (Salary), an RMSE of **`7059.04`** means, on average, our model's predictions are off by about $7,059.04. Considering the salary range in our dataset (`$37,732` to `$122,392`), this is a reasonable error. It tells us the typical deviation of our predictions from the actual salaries.

*   **R-squared (R2 Score):** The R-squared value of **`0.9024`** is very strong! This means that approximately **90.24% of the variance in salary can be explained by our feature** (`YearsExperience`) according to our model. An R-squared close to 1 indicates a very strong fit, suggesting that our model is doing a great job at explaining and predicting salaries based on experience in this dataset. This metric confirms that the model has captured a significant portion of the underlying relationship between experience and salary, performing well on this specific dataset.

## Model Comparison and Selection: Simple vs. Polynomial Regression

To ensure we select the most appropriate model, we will compare the simple linear regression model (using only `YearsExperience`) with a polynomial regression model (including `YearsExperience` and `YearsExperience^2`). This comparison will utilize a robust validation technique: **5-fold cross-validation**.

We will use `Pipelines` to streamline the process, combining `StandardScaler` (for feature scaling) with the regression model. Cross-validation will help us assess how well each model generalizes to unseen data, providing a more reliable estimate of performance than a single train-test split.

### Model Selection Criteria:
*   We will prefer the polynomial model **only if it achieves a meaningfully lower average cross-validated RMSE** compared to the simple model. A 'meaningful' difference often implies a significant improvement that justifies the increased model complexity.
*   If the improvement is marginal or non-existent, we will **prefer the simpler model** due to its ease of interpretation and lower risk of overfitting, following the principle of Occam's Razor (simpler explanations are generally better).

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define X and y from the cleaned salary_df
X = salary_df[['YearsExperience']]
y = salary_df['Salary']

# Define KFold for reproducible cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("--- Evaluating Simple Linear Regression Model (YearsExperience) ---")
simple_model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

# Evaluate using cross-validation
simple_rmse_scores = -cross_val_score(simple_model_pipeline, X, y, cv=kf, scoring='neg_root_mean_squared_error')
simple_r2_scores = cross_val_score(simple_model_pipeline, X, y, cv=kf, scoring='r2')

avg_simple_rmse = np.mean(simple_rmse_scores)
avg_simple_r2 = np.mean(simple_r2_scores)

print(f"Average RMSE (Simple Model): {avg_simple_rmse:.2f}")
print(f"Average R2 (Simple Model): {avg_simple_r2:.4f}")
print("-" * 60)

print("\n--- Evaluating Polynomial Regression Model (YearsExperience, YearsExperience^2) ---")
poly_model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly_features', PolynomialFeatures(degree=2, include_bias=False)),
    ('regressor', LinearRegression())
])

# Evaluate using cross-validation
poly_rmse_scores = -cross_val_score(poly_model_pipeline, X, y, cv=kf, scoring='neg_root_mean_squared_error')
poly_r2_scores = cross_val_score(poly_model_pipeline, X, y, cv=kf, scoring='r2')

avg_poly_rmse = np.mean(poly_rmse_scores)
avg_poly_r2 = np.mean(poly_r2_scores)

print(f"Average RMSE (Polynomial Model): {avg_poly_rmse:.2f}")
print(f"Average R2 (Polynomial Model): {avg_poly_r2:.4f}")
print("-" * 60)

print("\n--- Model Selection ---")
# Define a threshold for "meaningfully lower" RMSE, e.g., 1% improvement
rmse_improvement_threshold = 0.01

if avg_poly_rmse < avg_simple_rmse * (1 - rmse_improvement_threshold):
    print(f"The Polynomial Model has an average RMSE of {avg_poly_rmse:.2f}, which is meaningfully lower than the Simple Model's {avg_simple_rmse:.2f} (more than {rmse_improvement_threshold*100:.0f}% improvement).")
    print("Considering the potential for better fit, the Polynomial Model (degree 2) is selected for its improved performance.")
    selected_model_pipeline = poly_model_pipeline
    selected_model_name = "Polynomial Regression (Degree 2)"
else:
    print(f"The Polynomial Model's average RMSE ({avg_poly_rmse:.2f}) is not meaningfully lower (less than {rmse_improvement_threshold*100:.0f}% improvement) than the Simple Model's ({avg_simple_rmse:.2f}).")
    print("Given its simplicity and lower risk of overfitting, the Simple Linear Regression Model is selected.")
    selected_model_pipeline = simple_model_pipeline
    selected_model_name = "Simple Linear Regression"

print(f"\nFinal Selected Model: {selected_model_name}")

# Train the selected model pipeline on the training data (original X_train, not yet scaled or poly transformed)
selected_model_pipeline.fit(X_train, y_train)

# Make predictions on the test data (original X_test, not yet scaled or poly transformed)
y_final_pred = selected_model_pipeline.predict(X_test)

print("\nDiagnostic plots will be generated for the selected model.")

### Diagnostic Visuals for the Selected Model

To further evaluate the performance and assumptions of our selected model, we will generate two key diagnostic plots:

1.  **Actual vs. Predicted Plot:** This plot shows the true values (`y_test`) against the model's predictions (`y_final_pred`). A good model will have points clustering closely around a 45-degree line, indicating that predicted values are close to actual values.
2.  **Residual Plot:** This plot displays the residuals (the difference between actual and predicted values: `y_test - y_final_pred`) against the predicted values (`y_final_pred`). For a good linear model, residuals should be randomly scattered around zero, with no discernible patterns. Patterns (e.g., a curve, a funnel shape) can indicate issues like non-linearity, heteroscedasticity, or omitted variables.

## Comprehensive Summary of Machine Learning Model Building Steps

Let's recap all the essential steps we've taken in building our model, understanding what we've learned from each, and why they are important for a robust machine learning workflow.

### 1. Data Loading and Initial Checks
*   **What we did:** We started by loading the `Salary_dataset.csv` into a pandas DataFrame called `salary_df`. We then performed initial checks like `salary_df.info()`, `salary_df.head()`, `salary_df.tail()`, `salary_df.isna().sum()`, and `salary_df.duplicated().sum()` to get a first look at the data.
*   **Key Findings:**
    *   The dataset contained 30 entries and 3 columns initially.
    *   We confirmed there were **no missing values** and **no duplicate rows**.
    *   An 'Unnamed: 0' column was identified, which often acts as a redundant index.
*   **Why this is important:** This foundational step ensures our data is loaded correctly and gives us a preliminary understanding of its structure and immediate quality issues.

### 2. Data Cleaning: Handling Redundant Columns
*   **What we did:** Based on the initial checks, we identified that the 'Unnamed: 0' column was merely a redundant index. We then dropped this column from our `salary_df`.
*   **Key Finding:** The dataset was cleaned by removing an artifact column that had no analytical value.
*   **Why this is important:** Removing redundant or irrelevant columns reduces noise, simplifies the dataset, and prevents potential issues in model training.

### 3. Univariate Analysis (Individual Variable Exploration)
*   **What we did:** We examined each numerical variable ('YearsExperience' and 'Salary') individually. This involved generating descriptive statistics (`.describe()`), creating histograms with KDE, and box plots for both columns.
*   **Key Findings:**
    *   **YearsExperience:** Showed a relatively **even distribution** (ranging from 1.2 to 10.6 years) with no significant outliers.
    *   **Salary:** Exhibited a **right-skewed distribution** (ranging from $37,732 to $122,392), meaning more individuals earn lower salaries, but also with no extreme outliers.
*   **Why this is important:** Understanding the individual characteristics of each variable helps us identify their distributions, central tendencies, spread, and potential issues (like skewness) that might require special handling before modeling.

### 4. Bivariate Analysis (Relationship between Two Variables)
*   **What we did:** We investigated the relationship between 'YearsExperience' and 'Salary' using a scatter plot and by calculating their Pearson Correlation Coefficient.
*   **Key Findings:**
    *   The scatter plot revealed a clear **upward, linear trend**, indicating that as years of experience increase, salary generally increases.
    *   The Pearson Correlation Coefficient was approximately **0.9782**, confirming a **very strong positive linear relationship**.
*   **Why this is important:** This step is crucial for understanding how our features relate to our target variable, guiding feature selection and informing our choice of model (e.g., linear models are suitable for linear relationships).

### 5. Multivariate Analysis (Relationships among Multiple Variables)
*   **What we did:** To get a more comprehensive view, we generated a correlation matrix heatmap and a pair plot including both 'YearsExperience' and 'Salary'.
*   **Key Findings:**
    *   The **heatmap visually reinforced the very strong positive correlation** (0.98), showing a clear positive association between experience and salary.
    *   The **pair plot** confirmed both the individual distributions (even for 'YearsExperience', right-skewed for 'Salary') and the strong linear relationship between the two variables.
*   **Why this is important:** Provides a holistic overview of variable relationships and distributions, solidifying our understanding of the dataset's structure.


### 6. Data Preparation: Define Features (X) and Target (y)
*   **What we did:** We explicitly separated our pre-processed features (the scaled 'YearsExperience') into `X` and our target variable ('Salary') into `y`.
*   **Why this is important:** This step formalizes the input-output structure required by virtually all supervised machine learning models, making the data ready for the next stages.

### 7. Train-Test Split
*   **What we did:** We divided our `X` and `y` data into four sets: `X_train`, `X_test`, `y_train`, and `y_test`. We used an 80/20 split (`test_size=0.2`) and set `random_state=42` for reproducibility.
*   **Key Findings:**
    *   `X_train` and `y_train` now contain **24 samples** (80% of the data), which the model will use for learning.
    *   `X_test` and `y_test` contain the remaining **6 samples** (20% of the data), reserved exclusively for evaluating the model.
*   **Why this is important:** This is a critical step to ensure that we evaluate our model's performance on *unseen data*. It helps us assess how well the model generalizes to new, real-world examples, preventing the common problem of overfitting (where a model performs well on training data but poorly on new data).

### 8. Model Training: Linear Regression
*   **What we did:** We initialized a `LinearRegression` model from `sklearn.linear_model` and then used its `.fit()` method to train it using our `X_train` and `y_train` datasets.
*   **Key Findings (from the trained model):**
    *   **Model Intercept (~$74,208.62$):** This is the model's estimated base salary when the scaled `YearsExperience` is zero. It represents the starting point of the salary prediction.
    *   **Coefficient for YearsExperience (~$27,151.54$):** This positive value indicates that, for every one-unit increase in *scaled* `YearsExperience`, the predicted salary is expected to increase by approximately $27,151.54. This confirms a strong positive impact of experience on salary.
*   **Why this is important:** This is the core learning phase where the model identifies and quantifies the mathematical relationships between the features and the target. The intercept and coefficients are the learned parameters that define the model's predictive equation.

### 9. Model Validation: Predictions and Evaluation
*   **What we did:** We used our trained Linear Regression model to make predictions (`y_pred`) on the `X_test` data. Then, we compared these predictions to the actual `y_test` values using several evaluation metrics: Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and R-squared ($R^2$).
*   **Key Findings (from model evaluation):**
    *   **Root Mean Squared Error (RMSE) = $7,059.04$**: This means that, on average, our model's salary predictions were off by approximately $7,059.04. Given the range of salaries in the dataset, this indicates a reasonable level of predictive accuracy for this dataset.
    *   **R-squared ($R^2$) = 0.9024**: This is a very strong result! It means that our model, using `YearsExperience` as a feature, can explain approximately **90.24% of the variation in salary**. An R-squared value close to 1 suggests that our model fits the data very well and is highly effective at predicting salary based on the given feature for this dataset.
*   **Why this is important:** Validation is the crucial step where we objectively measure how well our model performs on new, unseen data. Good evaluation metrics indicate that our model is not only learning from the training data but can also generalize effectively to make accurate predictions for this specific dataset. This confirms the model's predictive power.

### 10. Report Generation
*   **What we did:** Finally, a comprehensive HTML business report (`index.html`) and a project README file (`README.md`) were generated. The HTML report incorporates key insights, model details, and embedded visualizations (histograms, box plots, scatter plots, correlation heatmaps, actual vs. predicted, and residual plots) for a clear and interactive summary of the project.
*   **Why this is important:** This step consolidates all project findings and visual evidence into easily shareable and reviewable documents, crucial for communicating results to stakeholders and for project documentation.